# Gradio API Server cho Demo Hướng B Fine-tuned

Notebook này chạy trên Colab để load:

- Model gốc: `Qwen/Qwen2.5-VL-3B-Instruct`
- Adapter fine-tuned trong Google Drive, ví dụ `final_finetuned_adapter`

Sau đó mở Gradio share link. Copy URL dạng `https://xxxx.gradio.live` và dán vào file local `demo/.env`:

```env
GRADIO_API_URL=https://xxxx.gradio.live
GRADIO_API_NAME=/predict
```

In [1]:
!pip -q install -U transformers accelerate peft bitsandbytes qwen-vl-utils gradio pillow

In [2]:
from pathlib import Path

import torch
import gradio as gr
from PIL import Image
from google.colab import drive
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel
from qwen_vl_utils import process_vision_info

drive.mount('/content/drive')

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Mounted at /content/drive
CUDA: True
GPU: Tesla T4


## Cấu hình path adapter

Nếu adapter của bạn nằm ở folder khác, sửa `ADAPTER_PATH` bên dưới. Theo ảnh Drive bạn gửi, adapter có thể nằm trong thư mục `final_finetuned_adapter`.

In [4]:
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'

PROJECT_ROOT = Path('/content/drive/MyDrive/Final_Deeplearning')

# Sửa dòng này nếu folder adapter của bạn khác.
ADAPTER_PATH = PROJECT_ROOT / 'qwen25_vl_herb_qlora_10chunks' / 'final_finetuned_adapter'

# Fallback thường gặp nếu bạn để adapter trực tiếp trong Final_Deeplearning.
if not ADAPTER_PATH.exists():
    alt = PROJECT_ROOT / 'final_finetuned_adapter'
    if alt.exists():
        ADAPTER_PATH = alt

assert ADAPTER_PATH.exists(), f'Không thấy adapter: {ADAPTER_PATH}'
print('Adapter:', ADAPTER_PATH)

Adapter: /content/drive/MyDrive/Final_Deeplearning/qwen25_vl_herb_qlora_10chunks/final_finetuned_adapter


In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH))
model.eval()
print('Loaded base model + fine-tuned adapter')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Loaded base model + fine-tuned adapter


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.visual.blocks.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.visual.blocks.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.visual.blocks.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.visual.blocks.0.mlp.up_proj.lora_B.default.weight', 'base_model.model.model.visual.blocks.0.mlp.down_proj.lora_A.default.weight', 'base_model.model.model.visual.blocks.0.mlp.down_proj.lora_B.default.weight', 'base_model.model.model.visual.blocks.1.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.visual.blocks.1.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.visual.blocks.1.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.visual.blocks.1.mlp.up_proj.lora_B.default.weight', 'base_model.model.model.visual.blocks.1.mlp.down_proj.lora_A.default.weight', 'base_model.mod

In [6]:
def build_prompt(question):
    return (
    'Bạn là hệ thống hỏi đáp ảnh dược liệu Việt Nam. '
    'Hãy trả lời câu hỏi bằng tiếng Việt, tối đa 10 từ. '
    'Ví dụ mẫu: Câu hỏi: Trong hình, lá đài phát triển có màu gì? trả lời: Màu trắng, mềm.\n'
    f'Câu hỏi: {question}'
)

@torch.inference_mode()
def predict(image_path, question):
    if image_path is None:
        return 'Vui l?ng upload ?nh.'
    if question is None or not str(question).strip():
        return 'Vui l?ng nh?p c?u h?i.'

    image_path = str(image_path)
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': image_path},
                {'type': 'text', 'text': build_prompt(str(question).strip())},
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt',
    )
    if torch.cuda.is_available():
        inputs = inputs.to('cuda')

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,
    )
    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    answer = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]
    return answer.strip()


In [8]:
demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Image(type='filepath', label='Ảnh dược liệu'),
        gr.Textbox(
            label='Câu hỏi tiếng Việt',
            placeholder='Hoa trong ảnh có màu gì?'
        ),
    ],
    outputs=gr.Textbox(label='Câu trả lời'),
    title='Vietnamese Medicinal Herb VQA - Qwen2.5-VL Fine-tuned',
    description='API endpoint dùng cho FastAPI demo local. Endpoint: /predict',
    api_name='predict',
)

demo.launch(share=True, debug=True, show_error=True)



Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cbaa87e36d1b327fb5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2173, in process_api
    inputs = await self.preprocess_data(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1788, in preprocess_data
    inputs_cached = data_model.model_validate(
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pydantic/main.py", line 716, in model_validate
    return cls.__pydantic_validator__.validate_python(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://cbaa87e36d1b327fb5.gradio.live
